In [1]:
import itertools
import multiprocessing
import os
import pathlib

import numpy as np
import pandas as pd
import tifffile
import tomli
from image_analysis_3D.file_utils.notebook_init_utils import (
    bandicoot_check,
    init_notebook,
)

root_dir, in_notebook = init_notebook()
if in_notebook:
    import tqdm.auto as tqdm
else:
    import tqdm
from tqdm import tqdm as tqdm_main

bandicoot_mount_path = pathlib.Path(os.path.expanduser("~/mnt/bandicoot"))
bandicoot_mount_path = bandicoot_check(bandicoot_mount_path, root_dir)

In [2]:
def change_mito_shape_to_match_nuclei(row):
    """
    If the mito image has a different shape than the nuclei image, copy the first z-slice and insert it at the beginning of the stack to make the shapes match.
    """
    if row.mito_image_shape != row.nuclei_image_shape:
        mito_image = tifffile.imread(row.mito_image_path)

        if row.mito_image_shape < row.nuclei_image_shape:
            # copy the first z-slice and insert it at the beginning of the stack
            first_slice = mito_image[0:1, :, :]
            new_mito_image = np.concatenate([first_slice, mito_image], axis=0)
            print(mito_image.shape, new_mito_image.shape)
        elif row.mito_image_shape > row.nuclei_image_shape:
            # remove the first z-slice to make the shapes match
            new_mito_image = mito_image[1:, :, :]
            print(mito_image.shape, new_mito_image.shape)

        # save the new mito image to a new path
        tifffile.imwrite(row.mito_image_path, new_mito_image)

In [3]:
patient = "NF0037_T1"

In [4]:
final_dict = {
    "patient": [],
    "well_fov": [],
    "mito_image_path": [],
    "dna_image_path": [],
}

patient_well_fovs = sorted(
    [
        path.name
        for path in (bandicoot_mount_path / "data" / patient / "zstack_images").glob(
            "*"
        )
        if path.is_dir()
    ]
)
for well_fov in tqdm.tqdm(
    patient_well_fovs, desc="Well/FOV", unit="well_fov", leave=False
):
    images = sorted(
        (bandicoot_mount_path / "data" / patient / "zstack_images" / well_fov).glob(
            "*.tif*"
        )
    )

    for image in images:
        if "640" in image.name:
            mito_image_path = image
        elif "405" in image.name:
            dna_image_path = image

    final_dict["patient"].append(patient)
    final_dict["well_fov"].append(well_fov)
    final_dict["mito_image_path"].append(mito_image_path)
    final_dict["dna_image_path"].append(dna_image_path)
df = pd.DataFrame(final_dict)
df

Well/FOV:   0%|          | 0/420 [00:00<?, ?well_fov/s]

,patient,well_fov,mito_image_path,dna_image_path
0,NF0037_T1,B10-1,/home/lippincm/mnt/bandicoot/NF1_organoid_data...,/home/lippincm/mnt/bandicoot/NF1_organoid_data...
1,NF0037_T1,B10-2,/home/lippincm/mnt/bandicoot/NF1_organoid_data...,/home/lippincm/mnt/bandicoot/NF1_organoid_data...
2,NF0037_T1,B10-3,/home/lippincm/mnt/bandicoot/NF1_organoid_data...,/home/lippincm/mnt/bandicoot/NF1_organoid_data...
3,NF0037_T1,B10-4,/home/lippincm/mnt/bandicoot/NF1_organoid_data...,/home/lippincm/mnt/bandicoot/NF1_organoid_data...
4,NF0037_T1,B10-5,/home/lippincm/mnt/bandicoot/NF1_organoid_data...,/home/lippincm/mnt/bandicoot/NF1_organoid_data...
...,...,...,...,...
415,NF0037_T1,G9-3,/home/lippincm/mnt/bandicoot/NF1_organoid_data...,/home/lippincm/mnt/bandicoot/NF1_organoid_data...
416,NF0037_T1,G9-4,/home/lippincm/mnt/bandicoot/NF1_organoid_data...,/home/lippincm/mnt/bandicoot/NF1_organoid_data...
417,NF0037_T1,G9-5,/home/lippincm/mnt/bandicoot/NF1_organoid_data...,/home/lippincm/mnt/bandicoot/NF1_organoid_data...
418,NF0037_T1,G9-6,/home/lippincm/mnt/bandicoot/NF1_organoid_data...,/home/lippincm/mnt/bandicoot/NF1_organoid_data...


In [5]:
mito_shapes = []
nuclei_shapes = []
for row in tqdm.tqdm(
    df.itertuples(), total=len(df), desc="Checking shapes", unit="well_fov"
):
    try:
        with tifffile.TiffFile(row.mito_image_path) as tif:
            shape = tif.series[0].shape
    except Exception as e:
        print(f"Error loading {image_path}: {e}")
        shape = None
    mito_shapes.append(shape)
    try:
        with tifffile.TiffFile(row.dna_image_path) as tif:
            shape = tif.series[0].shape
    except Exception as e:
        print(f"Error loading {image_path}: {e}")
        shape = None
    nuclei_shapes.append(shape)
df["mito_image_shape"] = mito_shapes
df["nuclei_image_shape"] = nuclei_shapes
df

Checking shapes:   0%|          | 0/420 [00:00<?, ?well_fov/s]

,patient,well_fov,mito_image_path,dna_image_path,mito_image_shape,nuclei_image_shape
0,NF0037_T1,B10-1,/home/lippincm/mnt/bandicoot/NF1_organoid_data...,/home/lippincm/mnt/bandicoot/NF1_organoid_data...,"(101, 1507, 1508)","(101, 1507, 1508)"
1,NF0037_T1,B10-2,/home/lippincm/mnt/bandicoot/NF1_organoid_data...,/home/lippincm/mnt/bandicoot/NF1_organoid_data...,"(101, 1507, 1508)","(101, 1507, 1508)"
2,NF0037_T1,B10-3,/home/lippincm/mnt/bandicoot/NF1_organoid_data...,/home/lippincm/mnt/bandicoot/NF1_organoid_data...,"(101, 1507, 1508)","(101, 1507, 1508)"
3,NF0037_T1,B10-4,/home/lippincm/mnt/bandicoot/NF1_organoid_data...,/home/lippincm/mnt/bandicoot/NF1_organoid_data...,"(101, 1507, 1508)","(101, 1507, 1508)"
4,NF0037_T1,B10-5,/home/lippincm/mnt/bandicoot/NF1_organoid_data...,/home/lippincm/mnt/bandicoot/NF1_organoid_data...,"(101, 1507, 1508)","(101, 1507, 1508)"
...,...,...,...,...,...,...
415,NF0037_T1,G9-3,/home/lippincm/mnt/bandicoot/NF1_organoid_data...,/home/lippincm/mnt/bandicoot/NF1_organoid_data...,"(101, 1507, 1508)","(101, 1507, 1508)"
416,NF0037_T1,G9-4,/home/lippincm/mnt/bandicoot/NF1_organoid_data...,/home/lippincm/mnt/bandicoot/NF1_organoid_data...,"(101, 1507, 1508)","(101, 1507, 1508)"
417,NF0037_T1,G9-5,/home/lippincm/mnt/bandicoot/NF1_organoid_data...,/home/lippincm/mnt/bandicoot/NF1_organoid_data...,"(101, 1507, 1508)","(101, 1507, 1508)"
418,NF0037_T1,G9-6,/home/lippincm/mnt/bandicoot/NF1_organoid_data...,/home/lippincm/mnt/bandicoot/NF1_organoid_data...,"(101, 1507, 1508)","(101, 1507, 1508)"


In [ ]:
for row in tqdm.tqdm(
    df.itertuples(), total=len(df), desc="Checking shapes", unit="row"
):
    change_mito_shape_to_match_nuclei(row)